# APPLICATION OF XGBOOST FROM APRIL 2026

In [ ]:
# # ============================================================
# # 🔍 PREDICT SECTION & CLUSTER — Device: LL1801
# # ============================================================

# import pandas as pd
# import numpy as np
# import re, pickle, os, warnings
# warnings.filterwarnings('ignore')

# # ── 1. LOAD MODELS & ENCODERS ────────────────────────────────
# SAVE_DIR = r'C:\Users\sitisyaziyah\Documents\Device_Identifier\Section\XGBoost16042026'

# with open(os.path.join(SAVE_DIR, 'model_section.pkl'), 'rb') as f:
#     model_section = pickle.load(f)

# with open(os.path.join(SAVE_DIR, 'model_cluster.pkl'), 'rb') as f:
#     model_cluster = pickle.load(f)

# with open(os.path.join(SAVE_DIR, 'label_encoders.pkl'), 'rb') as f:
#     enc           = pickle.load(f)

# le_customer    = enc['customer']
# le_project     = enc['project']
# le_prefix      = enc['prefix']
# le_suffix_lt   = enc['suffix_lt']
# le_suffix_last = enc['suffix_last']
# le_section     = enc['section']
# le_cluster     = enc['cluster']
# feature_cols   = enc['feature_columns']

# print(f"✅ Models loaded | Sections: {list(le_section.classes_)} | Clusters: {list(le_cluster.classes_)}")

# # ── 2. FEATURE ENGINEERING HELPERS ──────────────────────────
# def extract_numeric_block(did):
#     m = re.search(r'\d+', str(did))
#     return int(m.group()) if m else -1

# def extract_prefix(did):
#     m = re.match(r'^([A-Za-z]+)', str(did))
#     return m.group(1).upper() if m else ''

# def extract_suffix_lt(did):
#     m = re.search(r'\d+([A-Za-z]*)$', str(did))
#     return m.group(1).upper() if m else ''

# def extract_suffix_full(did):
#     m = re.search(r'\d+(.*)$', str(did))
#     return m.group(1) if m else ''

# def safe_encode(encoder, value, default=0):
#     try:
#         return int(encoder.transform([value])[0])
#     except ValueError:
#         print(f"  ⚠ Unseen label '{value}' → using default={default}")
#         return default

# import pandas as pd

# df_oiltek_num = pd.read_pickle(
#     r"C:\Users\sitisyaziyah\Documents\Device_Identifier\Section\XGBoost16042026\df_oiltek_num.pkl"
# )

# # ── 3. INPUT ─────────────────────────────────────────────────
# device_id = 'LL510'
# customer  = 'OILTEK'
# project   = 'A1706'

# # ── 4. COMPUTE numeric_block_rank (PURE NUMERIC COMPARISON) ──
# # ── 4. COMPUTE numeric_block_rank (FROM df_oiltek_num) ───────

# numeric_block = extract_numeric_block(device_id)

# # Filter relevant subset
# df_ref = df_oiltek_num[
#     (df_oiltek_num["CUSTOMER"] == customer) &
#     (df_oiltek_num["PROJECT"] == "A1706")
# ]

# if not df_ref.empty:
#     # Ensure numeric_block column is clean
#     df_ref = df_ref.copy()
#     df_ref["numeric_block"] = pd.to_numeric(df_ref["numeric_block"], errors="coerce")

#     # Get sorted unique numeric blocks
#     unique_blocks = sorted(df_ref["numeric_block"].dropna().unique())

#     # PURE ranking
#     numeric_block_rank = sum(b <= numeric_block for b in unique_blocks)

#     # Edge case: if all values are bigger
#     if numeric_block_rank == 0:
#         numeric_block_rank = 1

#     print(f"  ✅ Numeric-based rank (df reference): {numeric_block} → rank {numeric_block_rank}")

# else:
#     numeric_block_rank = 1
#     print("  ⚠ No reference data found — using rank=1 (fallback)")

✅ Models loaded | Sections: ['SECTION 1', 'SECTION 2', 'SECTION 3', 'SECTION 4', 'SECTION 5', 'SECTION 6'] | Clusters: ['CLUSTER 1', 'CLUSTER 2', 'CLUSTER 3', 'CLUSTER 4', 'CLUSTER 5']
  ✅ Numeric-based rank (df reference): 510 → rank 6


In [ ]:
# # ── 5. BUILD FEATURE VECTOR ──────────────────────────────────
# pfx        = extract_prefix(device_id)
# sfx_lt     = extract_suffix_lt(device_id)
# sfx_full   = extract_suffix_full(device_id)
# sfx_last   = sfx_full[-1] if sfx_full else ''


# # ── 5. BUILD FEATURE VECTOR (NUMERIC-ONLY VERSION) ───────────
# fv = {
#     'customer_enc'       : safe_encode(le_customer, customer),
#     'project_enc'        : safe_encode(le_project, project),
#     'numeric_block'      : numeric_block,
#     'numeric_block_rank' : numeric_block_rank,
#     'device_id_length'   : len(device_id),
#     'has_numeric'        : int(numeric_block != -1),
#     'equip_id_length'    : len(device_id),
#     'equip_id_digit_count': len(re.findall(r'\d', device_id)),
#     'equip_id_letter_count': len(re.findall(r'[A-Za-z]', device_id)),
# }

# X_new = pd.DataFrame([fv])

# X_new = pd.DataFrame([fv])

# # Ensure all expected columns exist
# for col in feature_cols:
#     if col not in X_new.columns:
#         X_new[col] = 0

# # Reorder columns
# X_new = X_new[feature_cols]


In [ ]:
# # ── 6. PREDICT ──────────────────────────────────────────────
# assert list(X_new.columns) == list(feature_cols), "❌ Feature mismatch with training!"

# try:
#     sec_proba = model_section.predict_proba(X_new)[0]
#     clu_proba = model_cluster.predict_proba(X_new)[0]
# except Exception as e:
#     print(f"❌ Prediction failed: {e}")
#     raise

# sec_pred = le_section.classes_[np.argmax(sec_proba)]
# clu_pred = le_cluster.classes_[np.argmax(clu_proba)]

# sec_conf = np.max(sec_proba)
# clu_conf = np.max(clu_proba)


# # ── 7. DISPLAY RESULTS ──────────────────────────────────────
# print("\n" + "=" * 65)
# print(f"  PREDICTION RESULT — {device_id} | {customer} | {project}")
# print("=" * 65)
# print(f"  ✅ Section  →  {sec_pred}   ({sec_conf:.2%})")
# print(f"  ✅ Cluster  →  {clu_pred}   ({clu_conf:.2%})")

# print("\n  SECTION probabilities:")
# sec_df = pd.DataFrame({'Section': le_section.classes_, 'Prob': sec_proba})
# sec_df = sec_df.sort_values('Prob', ascending=False).reset_index(drop=True)

# for _, row in sec_df.iterrows():
#     bar = '█' * int(row['Prob'] * 40)
#     tag = '<< PREDICTED' if row['Section'] == sec_pred else ''
#     print(f"    {row['Section']:<12} {row['Prob']:6.2%}  {bar} {tag}")

# print("\n  CLUSTER probabilities:")
# clu_df = pd.DataFrame({'Cluster': le_cluster.classes_, 'Prob': clu_proba})
# clu_df = clu_df.sort_values('Prob', ascending=False).reset_index(drop=True)

# for _, row in clu_df.iterrows():
#     bar = '█' * int(row['Prob'] * 40)
#     tag = '<< PREDICTED' if row['Cluster'] == clu_pred else ''
#     print(f"    {row['Cluster']:<12} {row['Prob']:6.2%}  {bar} {tag}")

# print("\n" + "=" * 65)


  PREDICTION RESULT — LL510 | OILTEK | A1706
  ✅ Section  →  SECTION 1   (65.80%)
  ✅ Cluster  →  CLUSTER 1   (99.48%)

  SECTION probabilities:
    SECTION 1    65.80%  ██████████████████████████ << PREDICTED
    SECTION 2    27.50%  ███████████ 
    SECTION 3     3.32%  █ 
    SECTION 4     1.17%   
    SECTION 5     1.16%   
    SECTION 6     1.05%   

  CLUSTER probabilities:
    CLUSTER 1    99.48%  ███████████████████████████████████████ << PREDICTED
    CLUSTER 3     0.21%   
    CLUSTER 4     0.16%   
    CLUSTER 5     0.08%   
    CLUSTER 2     0.07%   



In [ ]:
# print(df_oiltek_num[(df_oiltek_num["PROJECT"] == "A1706")& (df_oiltek_num["SECTION"] == "SECTION 1") & (df_oiltek_num["CLUSTER"] == "CLUSTER 1")])

  CUSTOMER PROJECT    SECTION    CLUSTER DEVICE_ID  numeric_block
0   OILTEK   A1706  SECTION 1  CLUSTER 1      A500            500
1   OILTEK   A1706  SECTION 1  CLUSTER 1     HT500            500
2   OILTEK   A1706  SECTION 1  CLUSTER 1    PT500A            500
3   OILTEK   A1706  SECTION 1  CLUSTER 1    PV500B            500
4   OILTEK   A1706  SECTION 1  CLUSTER 1     TE500            500


# NEW SCRIPT

In [12]:
# ============================================================
# 🤖 DEVICE CLUSTER — MODEL APPLICATION SCRIPT
# ============================================================
# Loads saved pipeline (models + encoders) from training phase
# and predicts SECTION + CLUSTER for new device entries.
#
# Threshold: confidence < 50% → labelled as UNKNOWN
# ============================================================

import pandas as pd
import numpy as np
import re
import pickle
import os
import warnings
warnings.filterwarnings('ignore')


# ============================================================
# 📂 PATHS — Update to match your saved model directory
# ============================================================
MODEL_DIR       = r"C:\Users\sitisyaziyah\source\repos\DeviceCluster\1.training_model\model_config"
MODEL_DIR_OUTPUT_PREDICTION = r"C:\Users\sitisyaziyah\source\repos\DeviceCluster\1.training_model\output_prediction"


UNKNOWN_THRESHOLD = 0.50   # confidence below this → UNKNOWN


# ============================================================
# 🔧 FEATURE ENGINEERING FUNCTIONS (must mirror training)
# ============================================================

def extract_numeric_block(device_id):
    match = re.search(r'\d+', str(device_id))
    return int(match.group()) if match else -1

def extract_prefix(device_id):
    match = re.match(r'^([A-Za-z]+)', str(device_id))
    return match.group(1).upper() if match else ''

def extract_suffix_letters(device_id):
    match = re.search(r'\d+([A-Za-z]*)$', str(device_id))
    return match.group(1).upper() if match else ''

def extract_suffix_full(device_id):
    match = re.search(r'\d+(.*)$', str(device_id))
    return match.group(1) if match else ''


# ============================================================
# 📦 LOAD PIPELINE
# ============================================================

def load_pipeline(model_dir: str) -> dict:
    """Load all saved models and encoders from training."""
    print("=" * 80)
    print("📦 LOADING SAVED PIPELINE")
    print("=" * 80)

    model_section   = pickle.load(open(os.path.join(model_dir, "model_section.pkl"),  "rb"))
    model_cluster   = pickle.load(open(os.path.join(model_dir, "model_cluster.pkl"),  "rb"))
    pipeline_config = pickle.load(open(os.path.join(model_dir, "pipeline_config.pkl"), "rb"))

    # Numeric block lookup table (used to compute rank for new data)
    df_num_path = os.path.join(model_dir, "df_oiltek_num.pkl")
    df_lookup   = pickle.load(open(df_num_path, "rb")) if os.path.exists(df_num_path) else None

    print("   ✅ model_section.pkl     loaded")
    print("   ✅ model_cluster.pkl     loaded")
    print("   ✅ pipeline_config.pkl   loaded")
    print(f"   {'✅' if df_lookup is not None else '⚠ '} df_oiltek_num.pkl       {'loaded' if df_lookup is not None else 'not found — rank will default to 1'}")
    print(f"\n   Section classes : {list(pipeline_config['le_section'].classes_)}")
    print(f"   Cluster classes : {list(pipeline_config['le_cluster'].classes_)}")

    return {
        "model_section"   : model_section,
        "model_cluster"   : model_cluster,
        "config"          : pipeline_config,
        "df_lookup"       : df_lookup,
    }


# ============================================================
# 🔧 BUILD FEATURES FOR ONE OR MORE DEVICES
# ============================================================

def build_features(records: list[dict], config: dict, df_lookup: pd.DataFrame | None) -> pd.DataFrame:
    """
    records : list of dicts with keys: device_id, customer, project
    Returns a DataFrame ready for model inference.
    """
    df = pd.DataFrame(records)
    df.columns = df.columns.str.upper()   # normalise column names

    le_customer     = config['le_customer']
    le_project      = config['le_project']
    le_prefix       = config['le_prefix']
    le_suffix_lt    = config['le_suffix_lt']
    le_suffix_last  = config['le_suffix_last']

    # ── BASIC DEVICE ID FEATURES ────────────────────────────
    df['numeric_block']          = df['DEVICE_ID'].apply(extract_numeric_block)
    df['device_prefix']          = df['DEVICE_ID'].apply(extract_prefix)
    df['device_suffix_letter']   = df['DEVICE_ID'].apply(extract_suffix_letters)
    df['suffix_full']            = df['DEVICE_ID'].apply(extract_suffix_full)
    df['prefix_length']          = df['device_prefix'].str.len()
    df['device_id_length']       = df['DEVICE_ID'].astype(str).str.len()
    df['has_suffix_letter']      = (df['device_suffix_letter'] != '').astype(int)
    df['has_numeric']            = (df['numeric_block'] != -1).astype(int)

    # ── SUFFIX FEATURES ─────────────────────────────────────
    df['suffix_length']             = df['suffix_full'].astype(str).str.len()
    df['suffix_has_digit']          = df['suffix_full'].astype(str).str.contains(r'\d', regex=True).astype(int)
    df['suffix_has_letter']         = df['suffix_full'].astype(str).str.contains(r'[A-Za-z]', regex=True).astype(int)
    df['suffix_has_decimal']        = df['suffix_full'].astype(str).str.contains(r'\.', regex=True).astype(int)
    df['suffix_digit_count']        = df['suffix_full'].astype(str).str.count(r'\d')
    df['suffix_letter_count']       = df['suffix_full'].astype(str).str.count(r'[A-Za-z]')
    df['suffix_starts_with_digit']  = df['suffix_full'].astype(str).str[0].str.isdigit().fillna(0).astype(int)
    df['suffix_last_char']          = df['suffix_full'].astype(str).str[-1]
    df['suffix_last_char_is_letter']= df['suffix_last_char'].str.isalpha().fillna(0).astype(int)
    df['suffix_last_char_is_digit'] = df['suffix_last_char'].str.isdigit().fillna(0).astype(int)

    # ── EQUIPMENT ID FEATURES ────────────────────────────────
    df['equip_id_length']       = df['DEVICE_ID'].astype(str).str.len()
    df['equip_id_digit_count']  = df['DEVICE_ID'].astype(str).str.count(r'\d')
    df['equip_id_letter_count'] = df['DEVICE_ID'].astype(str).str.count(r'[A-Za-z]')

    # ── NUMERIC BLOCK RANK ──────────────────────────────────
    # Compute rank using training lookup; fall back to 1 if not found
    def get_block_rank(row):
        if df_lookup is None:
            return 1
        subset = df_lookup[
            (df_lookup['CUSTOMER'] == row['CUSTOMER']) &
            (df_lookup['PROJECT']  == row['PROJECT'])
        ]
        if subset.empty:
            return 1
        unique_blocks = sorted(subset['numeric_block'].unique())
        block_to_rank = {b: i + 1 for i, b in enumerate(unique_blocks)}
        return block_to_rank.get(row['numeric_block'], 1)

    df['numeric_block_rank'] = df.apply(get_block_rank, axis=1)

    # ── LABEL ENCODING (handle unseen labels gracefully) ────
    def safe_encode(le, values):
        known = set(le.classes_)
        return np.array([
            le.transform([v])[0] if v in known else 0
            for v in values
        ])

    df['customer_enc']         = safe_encode(le_customer,    df['CUSTOMER'])
    df['project_enc']          = safe_encode(le_project,     df['PROJECT'])
    df['prefix_enc']           = safe_encode(le_prefix,      df['device_prefix'])
    df['suffix_letter_enc']    = safe_encode(le_suffix_lt,   df['device_suffix_letter'])
    df['suffix_last_char_enc'] = safe_encode(le_suffix_last, df['suffix_last_char'])

    return df



In [ ]:

# ============================================================
# 🚀 PREDICT
# ============================================================

def predict(records: list[dict], pipeline: dict, threshold: float = UNKNOWN_THRESHOLD) -> pd.DataFrame:
    """
    Runs the chained Section → Cluster prediction.

    Parameters
    ----------
    records   : list of dicts  [{device_id, customer, project}, ...]
    pipeline  : dict returned by load_pipeline()
    threshold : float — confidence below this → UNKNOWN

    Returns
    -------
    pd.DataFrame with prediction results and confidence scores.
    """
    config        = pipeline["config"]
    model_section = pipeline["model_section"]
    model_cluster = pipeline["model_cluster"]
    df_lookup     = pipeline["df_lookup"]
    le_section    = config["le_section"]
    le_cluster    = config["le_cluster"]

    # ── Build feature matrix ─────────────────────────────────
    df_feat = build_features(records, config, df_lookup)

    section_features = config["section_features"]
    cluster_features = config["cluster_features"]   # includes 'predicted_section'

    # Ensure only numeric cols used in training are present
    X_section = df_feat[section_features].apply(pd.to_numeric, errors='coerce').fillna(0)

    # ── Stage 1: Section prediction ──────────────────────────
    sec_proba    = model_section.predict_proba(X_section)          # shape (n, n_section_classes)
    sec_pred_enc = np.argmax(sec_proba, axis=1)
    sec_conf     = sec_proba.max(axis=1)

    # ── Chain: inject predicted section as feature ───────────
    X_cluster = X_section.copy()
    # Add predicted_section column (same name as training)
    X_cluster = df_feat[
        [f for f in cluster_features if f != 'predicted_section']
    ].apply(pd.to_numeric, errors='coerce').fillna(0)
    X_cluster["predicted_section"] = sec_pred_enc
    # Reorder to match training order
    X_cluster = X_cluster[cluster_features]

    # ── Stage 2: Cluster prediction ──────────────────────────
    clu_proba    = model_cluster.predict_proba(X_cluster)
    clu_pred_enc = np.argmax(clu_proba, axis=1)
    clu_conf     = clu_proba.max(axis=1)

    # ── Decode labels ────────────────────────────────────────
    sec_labels = le_section.inverse_transform(sec_pred_enc)
    clu_labels = le_cluster.inverse_transform(clu_pred_enc)

    # ── Apply UNKNOWN threshold ───────────────────────────────
    sec_final = np.where(sec_conf >= threshold, sec_labels, "UNKNOWN")
    clu_final = np.where(clu_conf >= threshold, clu_labels, "UNKNOWN")

    # ── Build result DataFrame ───────────────────────────────
    result = pd.DataFrame(records)
    result.columns = result.columns.str.upper()

    result["PREDICTED_SECTION"]    = sec_final
    result["SECTION_CONFIDENCE"]   = np.round(sec_conf * 100, 2)
    result["PREDICTED_CLUSTER"]    = clu_final
    result["CLUSTER_CONFIDENCE"]   = np.round(clu_conf * 100, 2)

    return result


# ============================================================
# 🖨️  PRETTY PRINT RESULTS
# ============================================================

def print_results(result: pd.DataFrame):
    print("\n" + "=" * 80)
    print("📊 PREDICTION RESULTS")
    print("=" * 80)

    for _, row in result.iterrows():
        sec_flag = "⚠ UNKNOWN" if row["PREDICTED_SECTION"] == "UNKNOWN" else "✅"
        clu_flag = "⚠ UNKNOWN" if row["PREDICTED_CLUSTER"] == "UNKNOWN" else "✅"

        print(f"\n   Device   : {row['DEVICE_ID']}")
        print(f"   Customer : {row['CUSTOMER']}   |   Project : {row['PROJECT']}")
        print(f"   {sec_flag} Section : {row['PREDICTED_SECTION']:20s}  (confidence: {row['SECTION_CONFIDENCE']:.1f}%)")
        print(f"   {clu_flag} Cluster : {row['PREDICTED_CLUSTER']:20s}  (confidence: {row['CLUSTER_CONFIDENCE']:.1f}%)")
        print(f"   {'-' * 60}")

    print(f"\n{'=' * 80}")
    print(f"   Total records predicted : {len(result)}")
    unknown_sec = (result["PREDICTED_SECTION"] == "UNKNOWN").sum()
    unknown_clu = (result["PREDICTED_CLUSTER"] == "UNKNOWN").sum()
    if unknown_sec or unknown_clu:
        print(f"   ⚠  UNKNOWN Section : {unknown_sec}   |   UNKNOWN Cluster : {unknown_clu}")
    print(f"   Confidence threshold used : {UNKNOWN_THRESHOLD * 100:.0f}%")
    print("=" * 80)


# ============================================================
# ▶️  MAIN — EXAMPLE APPLICATION
# ============================================================

if __name__ == "__main__":

    # ----------------------------------------------------------
    # 🔹 EXAMPLE INPUT DATA
    #    Format: list of dicts with keys: device_id, customer, project
    # ----------------------------------------------------------
    test_records = [
        # ── Single device (your provided example) ──
        {"device_id": "LL510",    "customer": "OILTEK", "project": "A1706"},

        # # ── More examples to stress-test the model ──
        {"device_id": "HT275",    "customer": "OILTEK", "project": "A1706"},
        # {"device_id": "CPM6311",  "customer": "OILTEK", "project": "A1706"},
        # {"device_id": "FIT151",   "customer": "OILTEK", "project": "A1706"},
        # {"device_id": "V29011",   "customer": "OILTEK", "project": "A1706"},
        # {"device_id": "PT500A",   "customer": "OILTEK", "project": "A1706"},

        # # ── Edge cases ──────────────────────────────────────
        {"device_id": "ZZZ9999X", "customer": "OILTEK", "project": "A1706"},   # unknown-looking device
        {"device_id": "LL510",    "customer": "UNKNOWN_CUSTOMER", "project": "UNKNOWN_PROJECT"},  # unseen customer/project
    ]

    # ----------------------------------------------------------
    # 1. Load pipeline
    # ----------------------------------------------------------
    pipeline = load_pipeline(MODEL_DIR)

    # ----------------------------------------------------------
    # 2. Predict
    # ----------------------------------------------------------
    result_df = predict(test_records, pipeline, threshold=UNKNOWN_THRESHOLD)

    # ----------------------------------------------------------
    # 3. Print results
    # ----------------------------------------------------------
    print_results(result_df)

    # ----------------------------------------------------------
    # 4. Optional: export to Excel
    # ----------------------------------------------------------
    out_path = os.path.join(MODEL_DIR_OUTPUT_PREDICTION, "application_results.xlsx")
    result_df.to_excel(out_path, index=False)
    print(f"\n   💾 Results saved to: {out_path}")


# ============================================================
# 🔁  SINGLE-DEVICE CONVENIENCE FUNCTION
#     Use this for quick one-off lookups in other scripts
# ============================================================

def predict_single(device_id: str, customer: str, project: str,
                   pipeline: dict, threshold: float = UNKNOWN_THRESHOLD) -> dict:
    """
    Shorthand wrapper for predicting a single device.

    Example
    -------
    >>> pipeline = load_pipeline(MODEL_DIR)
    >>> result = predict_single("LL510", "OILTEK", "A1706", pipeline)
    >>> print(result)
    """
    result = predict(
        [{"device_id": device_id, "customer": customer, "project": project}],
        pipeline,
        threshold=threshold
    )
    row = result.iloc[0]
    return {
        "device_id"           : row["DEVICE_ID"],
        "customer"            : row["CUSTOMER"],
        "project"             : row["PROJECT"],
        "predicted_section"   : row["PREDICTED_SECTION"],
        "section_confidence"  : row["SECTION_CONFIDENCE"],
        "predicted_cluster"   : row["PREDICTED_CLUSTER"],
        "cluster_confidence"  : row["CLUSTER_CONFIDENCE"],
    }

# output is not satisfy. need to focus on how unknown customer still manage to catch up the prediction. it shouldn't supposedly

📦 LOADING SAVED PIPELINE
   ✅ model_section.pkl     loaded
   ✅ model_cluster.pkl     loaded
   ✅ pipeline_config.pkl   loaded
   ✅ df_oiltek_num.pkl       loaded

   Section classes : ['SECTION 1', 'SECTION 2', 'SECTION 3', 'SECTION 4', 'SECTION 5', 'SECTION 6']
   Cluster classes : ['CLUSTER 1', 'CLUSTER 2', 'CLUSTER 3', 'CLUSTER 4', 'CLUSTER 5', 'CLUSTER 6', 'CLUSTER 7', 'CLUSTER 8', 'CLUSTER 9']

📊 PREDICTION RESULTS

   Device   : LL510
   Customer : OILTEK   |   Project : A1706
   ✅ Section : SECTION 1             (confidence: 74.8%)
   ✅ Cluster : CLUSTER 1             (confidence: 98.7%)
   ------------------------------------------------------------

   Device   : HT275
   Customer : OILTEK   |   Project : A1706
   ✅ Section : SECTION 3             (confidence: 93.9%)
   ✅ Cluster : CLUSTER 1             (confidence: 87.9%)
   ------------------------------------------------------------

   Device   : ZZZ9999X
   Customer : OILTEK   |   Project : A1706
   ✅ Section : SECTION 6

# EDIT NEW SCRIPT

In [14]:
# ============================================================
# 🤖 DEVICE CLUSTER — MODEL APPLICATION SCRIPT
# ============================================================
# Loads saved pipeline (models + encoders) from training phase
# and predicts SECTION + CLUSTER for new device entries.
#
# Guard rules (evaluated BEFORE model inference):
#   • Unseen CUSTOMER          → BLOCKED (confidence: N/A)
#   • Unseen PROJECT           → BLOCKED (confidence: N/A)
#   • Confidence < threshold   → UNKNOWN (shown with actual %)
# ============================================================

import pandas as pd
import numpy as np
import re
import pickle
import os
import warnings
warnings.filterwarnings('ignore')


# ============================================================
# 📂 PATHS — Update to match your saved model directory
# ============================================================
MODEL_DIR = r"C:\Users\sitisyaziyah\source\repos\DeviceCluster\1.training_model\model_config"

UNKNOWN_THRESHOLD = 0.50   # confidence below this → UNKNOWN


# ============================================================
# 🔧 FEATURE ENGINEERING FUNCTIONS (must mirror training)
# ============================================================

def extract_numeric_block(device_id):
    match = re.search(r'\d+', str(device_id))
    return int(match.group()) if match else -1

def extract_prefix(device_id):
    match = re.match(r'^([A-Za-z]+)', str(device_id))
    return match.group(1).upper() if match else ''

def extract_suffix_letters(device_id):
    match = re.search(r'\d+([A-Za-z]*)$', str(device_id))
    return match.group(1).upper() if match else ''

def extract_suffix_full(device_id):
    match = re.search(r'\d+(.*)$', str(device_id))
    return match.group(1) if match else ''


# ============================================================
# 📦 LOAD PIPELINE
# ============================================================

def load_pipeline(model_dir: str) -> dict:
    """Load all saved models and encoders from training."""
    print("=" * 80)
    print("📦 LOADING SAVED PIPELINE")
    print("=" * 80)

    model_section   = pickle.load(open(os.path.join(model_dir, "model_section.pkl"),  "rb"))
    model_cluster   = pickle.load(open(os.path.join(model_dir, "model_cluster.pkl"),  "rb"))
    pipeline_config = pickle.load(open(os.path.join(model_dir, "pipeline_config.pkl"), "rb"))

    df_num_path = os.path.join(model_dir, "df_oiltek_num.pkl")
    df_lookup   = pickle.load(open(df_num_path, "rb")) if os.path.exists(df_num_path) else None

    print("   ✅ model_section.pkl     loaded")
    print("   ✅ model_cluster.pkl     loaded")
    print("   ✅ pipeline_config.pkl   loaded")
    print(f"   {'✅' if df_lookup is not None else '⚠ '} df_oiltek_num.pkl       "
          f"{'loaded' if df_lookup is not None else 'not found — rank will default to 1'}")
    print(f"\n   Known customers : {sorted(pipeline_config['le_customer'].classes_)}")
    print(f"   Known projects  : {sorted(pipeline_config['le_project'].classes_)}")
    print(f"   Section classes : {list(pipeline_config['le_section'].classes_)}")
    print(f"   Cluster classes : {list(pipeline_config['le_cluster'].classes_)}")

    return {
        "model_section" : model_section,
        "model_cluster" : model_cluster,
        "config"        : pipeline_config,
        "df_lookup"     : df_lookup,
    }


# ============================================================
# 🛡️  GUARD — DETECT UNSEEN ENTITIES BEFORE INFERENCE
# ============================================================

def check_unseen(df: pd.DataFrame, config: dict) -> pd.Series:
    """
    Returns a Series of rejection reasons per row.
    Empty string = record is eligible for inference.

    WHY this matters
    ----------------
    LabelEncoder silently maps unseen values to index 0, which
    resolves to a real known entity and produces confident but
    completely meaningless predictions.  This guard stops that.
    """
    known_customers = set(config['le_customer'].classes_)
    known_projects  = set(config['le_project'].classes_)

    reasons = []
    for _, row in df.iterrows():
        row_reasons = []
        if row['CUSTOMER'] not in known_customers:
            row_reasons.append(f"unseen CUSTOMER '{row['CUSTOMER']}'")
        if row['PROJECT'] not in known_projects:
            row_reasons.append(f"unseen PROJECT '{row['PROJECT']}'")
        reasons.append("; ".join(row_reasons))

    return pd.Series(reasons, index=df.index)


# ============================================================
# 🔧 BUILD FEATURES (eligible rows only)
# ============================================================

def build_features(df: pd.DataFrame, config: dict, df_lookup) -> pd.DataFrame:
    """
    df must already be guard-checked (all customers/projects known).
    prefix/suffix letters may still be unseen → safely fall back to 0.
    """
    le_customer    = config['le_customer']
    le_project     = config['le_project']
    le_prefix      = config['le_prefix']
    le_suffix_lt   = config['le_suffix_lt']
    le_suffix_last = config['le_suffix_last']

    df = df.copy()

    # ── BASIC DEVICE ID FEATURES ────────────────────────────
    df['numeric_block']         = df['DEVICE_ID'].apply(extract_numeric_block)
    df['device_prefix']         = df['DEVICE_ID'].apply(extract_prefix)
    df['device_suffix_letter']  = df['DEVICE_ID'].apply(extract_suffix_letters)
    df['suffix_full']           = df['DEVICE_ID'].apply(extract_suffix_full)
    df['prefix_length']         = df['device_prefix'].str.len()
    df['device_id_length']      = df['DEVICE_ID'].astype(str).str.len()
    df['has_suffix_letter']     = (df['device_suffix_letter'] != '').astype(int)
    df['has_numeric']           = (df['numeric_block'] != -1).astype(int)

    # ── SUFFIX FEATURES ─────────────────────────────────────
    df['suffix_length']             = df['suffix_full'].astype(str).str.len()
    df['suffix_has_digit']          = df['suffix_full'].astype(str).str.contains(r'\d', regex=True).astype(int)
    df['suffix_has_letter']         = df['suffix_full'].astype(str).str.contains(r'[A-Za-z]', regex=True).astype(int)
    df['suffix_has_decimal']        = df['suffix_full'].astype(str).str.contains(r'\.', regex=True).astype(int)
    df['suffix_digit_count']        = df['suffix_full'].astype(str).str.count(r'\d')
    df['suffix_letter_count']       = df['suffix_full'].astype(str).str.count(r'[A-Za-z]')
    df['suffix_starts_with_digit']  = df['suffix_full'].astype(str).str[0].str.isdigit().fillna(0).astype(int)
    df['suffix_last_char']          = df['suffix_full'].astype(str).str[-1]
    df['suffix_last_char_is_letter']= df['suffix_last_char'].str.isalpha().fillna(0).astype(int)
    df['suffix_last_char_is_digit'] = df['suffix_last_char'].str.isdigit().fillna(0).astype(int)

    # ── EQUIPMENT ID FEATURES ────────────────────────────────
    df['equip_id_length']       = df['DEVICE_ID'].astype(str).str.len()
    df['equip_id_digit_count']  = df['DEVICE_ID'].astype(str).str.count(r'\d')
    df['equip_id_letter_count'] = df['DEVICE_ID'].astype(str).str.count(r'[A-Za-z]')

    # ── NUMERIC BLOCK RANK ───────────────────────────────────
    def get_block_rank(row):
        if df_lookup is None:
            return 1
        subset = df_lookup[
            (df_lookup['CUSTOMER'] == row['CUSTOMER']) &
            (df_lookup['PROJECT']  == row['PROJECT'])
        ]
        if subset.empty:
            return 1
        unique_blocks = sorted(subset['numeric_block'].unique())
        block_to_rank = {b: i + 1 for i, b in enumerate(unique_blocks)}
        return block_to_rank.get(row['numeric_block'], 1)

    df['numeric_block_rank'] = df.apply(get_block_rank, axis=1)

    # ── LABEL ENCODING ──────────────────────────────────────
    # customer/project: guaranteed known at this point → direct transform
    # prefix/suffix: may be unseen → safe fallback to 0
    def safe_encode(le, values):
        known = set(le.classes_)
        return np.array([
            le.transform([v])[0] if v in known else 0
            for v in values
        ])

    df['customer_enc']         = le_customer.transform(df['CUSTOMER'])
    df['project_enc']          = le_project.transform(df['PROJECT'])
    df['prefix_enc']           = safe_encode(le_prefix,      df['device_prefix'])
    df['suffix_letter_enc']    = safe_encode(le_suffix_lt,   df['device_suffix_letter'])
    df['suffix_last_char_enc'] = safe_encode(le_suffix_last, df['suffix_last_char'])

    return df


# ============================================================
# 🚀 PREDICT
# ============================================================

def predict(records: list[dict], pipeline: dict,
            threshold: float = UNKNOWN_THRESHOLD) -> pd.DataFrame:
    """
    Guard check first → model inference only for eligible rows.

    Returns
    -------
    pd.DataFrame with columns:
        DEVICE_ID, CUSTOMER, PROJECT,
        PREDICTED_SECTION, SECTION_CONFIDENCE (%),
        PREDICTED_CLUSTER, CLUSTER_CONFIDENCE (%),
        REJECTION_REASON   (empty string if eligible)
    """
    config        = pipeline["config"]
    model_section = pipeline["model_section"]
    model_cluster = pipeline["model_cluster"]
    df_lookup     = pipeline["df_lookup"]
    le_section    = config["le_section"]
    le_cluster    = config["le_cluster"]

    # ── Normalise input ──────────────────────────────────────
    base_df = pd.DataFrame(records)
    base_df.columns = base_df.columns.str.upper()

    # ── Guard ────────────────────────────────────────────────
    rejection_reasons = check_unseen(base_df, config)
    eligible_mask     = rejection_reasons == ""
    blocked_mask      = ~eligible_mask

    # ── Pre-fill all rows as UNKNOWN / N/A ───────────────────
    pred_section = ["UNKNOWN"] * len(base_df)
    sec_conf     = [None]      * len(base_df)   # None = not evaluated
    pred_cluster = ["UNKNOWN"] * len(base_df)
    clu_conf     = [None]      * len(base_df)

    # ── Run inference on eligible rows only ──────────────────
    if eligible_mask.any():
        elig_idx  = base_df.index[eligible_mask].tolist()
        df_elig   = base_df.loc[elig_idx].copy().reset_index(drop=True)
        df_feat   = build_features(df_elig, config, df_lookup)

        section_features = config["section_features"]
        cluster_features = config["cluster_features"]

        X_sec = df_feat[section_features].apply(pd.to_numeric, errors='coerce').fillna(0)

        # Stage 1 — Section
        sec_proba_raw = model_section.predict_proba(X_sec)
        sec_pred_enc  = np.argmax(sec_proba_raw, axis=1)
        sec_conf_raw  = sec_proba_raw.max(axis=1)

        # Chain
        X_clu = df_feat[
            [f for f in cluster_features if f != 'predicted_section']
        ].apply(pd.to_numeric, errors='coerce').fillna(0).copy()
        X_clu["predicted_section"] = sec_pred_enc
        X_clu = X_clu[cluster_features]

        # Stage 2 — Cluster
        clu_proba_raw = model_cluster.predict_proba(X_clu)
        clu_pred_enc  = np.argmax(clu_proba_raw, axis=1)
        clu_conf_raw  = clu_proba_raw.max(axis=1)

        # Decode
        sec_labels = le_section.inverse_transform(sec_pred_enc)
        clu_labels = le_cluster.inverse_transform(clu_pred_enc)

        # Apply threshold
        sec_final = np.where(sec_conf_raw >= threshold, sec_labels, "UNKNOWN")
        clu_final = np.where(clu_conf_raw >= threshold, clu_labels, "UNKNOWN")

        # Write back to original positions
        for i, orig_idx in enumerate(elig_idx):
            pred_section[orig_idx] = sec_final[i]
            sec_conf[orig_idx]     = round(float(sec_conf_raw[i]) * 100, 2)
            pred_cluster[orig_idx] = clu_final[i]
            clu_conf[orig_idx]     = round(float(clu_conf_raw[i]) * 100, 2)

    # ── Assemble result ──────────────────────────────────────
    result = base_df[["DEVICE_ID", "CUSTOMER", "PROJECT"]].copy()
    result["PREDICTED_SECTION"]  = pred_section
    result["SECTION_CONFIDENCE"] = sec_conf       # None for blocked rows
    result["PREDICTED_CLUSTER"]  = pred_cluster
    result["CLUSTER_CONFIDENCE"] = clu_conf       # None for blocked rows
    result["REJECTION_REASON"]   = rejection_reasons.values

    return result


# ============================================================
# 🖨️  PRETTY PRINT RESULTS
# ============================================================

def print_results(result: pd.DataFrame):
    print("\n" + "=" * 80)
    print("📊 PREDICTION RESULTS")
    print("=" * 80)

    for _, row in result.iterrows():
        rejected = bool(row["REJECTION_REASON"])
        print(f"\n   Device   : {row['DEVICE_ID']}")
        print(f"   Customer : {row['CUSTOMER']}   |   Project : {row['PROJECT']}")

        if rejected:
            print(f"   🚫 BLOCKED  — {row['REJECTION_REASON']}")
            print(f"   ⛔ Section : {'UNKNOWN':20s}  (confidence: N/A — not evaluated)")
            print(f"   ⛔ Cluster : {'UNKNOWN':20s}  (confidence: N/A — not evaluated)")
        else:
            sec_unk  = row["PREDICTED_SECTION"] == "UNKNOWN"
            clu_unk  = row["PREDICTED_CLUSTER"] == "UNKNOWN"
            sec_icon = "⚠" if sec_unk else "✅"
            clu_icon = "⚠" if clu_unk else "✅"
            sec_conf_str = f"{row['SECTION_CONFIDENCE']:.1f}%" if row["SECTION_CONFIDENCE"] is not None else "N/A"
            clu_conf_str = f"{row['CLUSTER_CONFIDENCE']:.1f}%" if row["CLUSTER_CONFIDENCE"] is not None else "N/A"

            print(f"   {sec_icon} Section : {row['PREDICTED_SECTION']:20s}  (confidence: {sec_conf_str})")
            print(f"   {clu_icon} Cluster : {row['PREDICTED_CLUSTER']:20s}  (confidence: {clu_conf_str})")

        print(f"   {'-' * 60}")

    # ── Summary ──────────────────────────────────────────────
    total     = len(result)
    blocked   = (result["REJECTION_REASON"] != "").sum()
    evaluated = total - blocked
    unk_sec   = ((result["PREDICTED_SECTION"] == "UNKNOWN") & (result["REJECTION_REASON"] == "")).sum()
    unk_clu   = ((result["PREDICTED_CLUSTER"] == "UNKNOWN") & (result["REJECTION_REASON"] == "")).sum()

    print(f"\n{'=' * 80}")
    print(f"   Total records              : {total}")
    print(f"   Evaluated by model         : {evaluated}   |   Blocked (unseen entity) : {blocked}")
    if unk_sec or unk_clu:
        print(f"   Low-confidence UNKNOWN  →  Section : {unk_sec}   |   Cluster : {unk_clu}")
    print(f"   Confidence threshold used  : {UNKNOWN_THRESHOLD * 100:.0f}%")
    print("=" * 80)


# ============================================================
# ▶️  MAIN — EXAMPLE APPLICATION
# ============================================================

if __name__ == "__main__":

    test_records = [
        # ── Your primary example ─────────────────────────────
        {"device_id": "LL510",    "customer": "OILTEK", "project": "A1706"},

        # ── From your notebook error analysis table ──────────
        {"device_id": "HT275",    "customer": "OILTEK", "project": "A1706"},
        {"device_id": "CPM6311",  "customer": "OILTEK", "project": "A1706"},
        {"device_id": "FIT151",   "customer": "OILTEK", "project": "A1706"},
        {"device_id": "V29011",   "customer": "OILTEK", "project": "A1706"},
        {"device_id": "PT500A",   "customer": "OILTEK", "project": "A1706"},

        # ── Edge: unusual device, known customer/project ─────
        {"device_id": "ZZZ9999X", "customer": "OILTEK",           "project": "A1706"},

        # ── Edge: unseen customer only → BLOCKED ─────────────
        {"device_id": "LL510",    "customer": "UNKNOWN_CUSTOMER", "project": "A1706"},

        # ── Edge: unseen project only → BLOCKED ──────────────
        {"device_id": "LL510",    "customer": "OILTEK",           "project": "UNKNOWN_PROJECT"},

        # ── Edge: both unseen → BLOCKED with combined reason ─
        {"device_id": "LL510",    "customer": "UNKNOWN_CUSTOMER", "project": "UNKNOWN_PROJECT"},
    ]

    # 1. Load
    pipeline = load_pipeline(MODEL_DIR)

    # 2. Predict
    result_df = predict(test_records, pipeline, threshold=UNKNOWN_THRESHOLD)

    # 3. Print
    print_results(result_df)

    # 4. Export
    out_path = os.path.join(MODEL_DIR, "application_results.xlsx")
    result_df.to_excel(out_path, index=False)
    print(f"\n   💾 Results saved to: {out_path}")


# ============================================================
# 🔁  SINGLE-DEVICE CONVENIENCE FUNCTION
# ============================================================

def predict_single(device_id: str, customer: str, project: str,
                   pipeline: dict, threshold: float = UNKNOWN_THRESHOLD) -> dict:
    """
    Shorthand for one device. Returns a plain dict.

    Example
    -------
    >>> pipeline = load_pipeline(MODEL_DIR)
    >>> result = predict_single("LL510", "OILTEK", "A1706", pipeline)
    >>> print(result)
    """
    result = predict(
        [{"device_id": device_id, "customer": customer, "project": project}],
        pipeline,
        threshold=threshold,
    )
    row = result.iloc[0]
    return {
        "device_id"          : row["DEVICE_ID"],
        "customer"           : row["CUSTOMER"],
        "project"            : row["PROJECT"],
        "predicted_section"  : row["PREDICTED_SECTION"],
        "section_confidence" : row["SECTION_CONFIDENCE"],   # None if blocked
        "predicted_cluster"  : row["PREDICTED_CLUSTER"],
        "cluster_confidence" : row["CLUSTER_CONFIDENCE"],   # None if blocked
        "rejection_reason"   : row["REJECTION_REASON"],
    }

📦 LOADING SAVED PIPELINE
   ✅ model_section.pkl     loaded
   ✅ model_cluster.pkl     loaded
   ✅ pipeline_config.pkl   loaded
   ✅ df_oiltek_num.pkl       loaded

   Known customers : ['OILTEK', 'UGS']
   Known projects  : ['A1706', 'A1993', 'A1994', 'A1995', 'A2027', 'A8881', 'A8882', 'A8883', 'A8884', 'A8885', 'A9991', 'A9992']
   Section classes : ['SECTION 1', 'SECTION 2', 'SECTION 3', 'SECTION 4', 'SECTION 5', 'SECTION 6']
   Cluster classes : ['CLUSTER 1', 'CLUSTER 2', 'CLUSTER 3', 'CLUSTER 4', 'CLUSTER 5', 'CLUSTER 6', 'CLUSTER 7', 'CLUSTER 8', 'CLUSTER 9']

📊 PREDICTION RESULTS

   Device   : LL510
   Customer : OILTEK   |   Project : A1706
   ✅ Section : SECTION 1             (confidence: 74.8%)
   ✅ Cluster : CLUSTER 1             (confidence: 98.7%)
   ------------------------------------------------------------

   Device   : HT275
   Customer : OILTEK   |   Project : A1706
   ✅ Section : SECTION 3             (confidence: 93.9%)
   ✅ Cluster : CLUSTER 1             (confi